# 🌲 Modelado de Regresión y Analítica de Datos: Optimización del Proceso de Blanqueo de Celulosa

## 🏭 Contexto Industrial & Planteamiento del Problema
La industria de celulosa en Chile opera sistemas avanzados de control para el blanqueo químico de pulpa Kraft en secuencias multietapa:
$$\text{Pre-Blanqueo} \longrightarrow \text{D}_0 \longrightarrow \text{EOP} \longrightarrow \text{D}_1 \longrightarrow \text{D}_2$$

- **Objetivo General**: Identificar las variables explicativas clave y construir el mejor modelo de regresión lineal para pronosticar y controlar el **Consumo Específico de Dióxido de Cloro ($ClO_2$)** en kg/ADt.
- **Límite Operacional de Diseño**: $17.50\text{ kg/ADt}$.
- **Consumo Promedio Histórico**: $19.10\text{ kg/ADt}$ (con picos hasta $20.37\text{ kg/ADt}$).
- **Impacto Económico**: 
  - Sobreconsumo evitable: **USD $1,600,000 / año**.
  - Beneficio potencial por optimización del 30%: **USD $500,000 / año**.

---
### ⚙️ Metodología y Objetivos Específicos
1. **Comprensión del Problema**: Formalización matemática, mapa de procesos y evaluación económica.
2. **Entendimiento de los Datos**: Clasificación de 100 sensores por dominio operativo y análisis exploratorio (EDA).
3. **Preparación de Datos**: Detección de varianza cero, prevención rigurosa de fuga de datos (*data leakage*), partición 80/20 (`random_state=2022`) y estandarización `StandardScaler`.
4. **Modelado y Diagnóstico**:
   - Regresión Lineal Clásica (MCO/OLS) + Diagnósticos Econométricos (Jarque-Bera, Durbin-Watson, Breusch-Pagan, Multicolinealidad).
   - Regresión Stepwise (optimizando Criterio de Información de Akaike - AIC).
   - Regresión Ridge ($L_2$) con $\lambda$ óptimo por validación cruzada.
   - Regresión Lasso ($L_1$) con $\lambda$ óptimo y selección esparsa.
   - Regresión Elastic Net ($L_1+L_2$) con $\lambda$ y $\alpha$ (l1-ratio) óptimos.
5. **Evaluación Comparativa & Selección del Modelo Óptimo**.


In [ ]:
import sys
from pathlib import Path

# Configurar path raíz del proyecto
ROOT_PATH = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_PATH))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CONFIG
from src.data.data_loader import IndustrialDataLoader
from src.data.data_preprocessor import IndustrialDataPreprocessor
from src.features.feature_engineering import DomainFeatureClassifier
from src.models.model_factory import ModelFactory
from src.models.model_evaluator import ModelEvaluator
from src.models.ols_regression import OLSRegressionModel
from src.models.stepwise_regression import StepwiseAICRegressionModel
from src.models.regularized_models import (
    RidgeRegressionModel,
    LassoRegressionModel,
    ElasticNetRegressionModel
)
from src.visualization.plotters import IndustrialVisualizer

print(f'[*] Configuración cargada con éxito. Semilla: {CONFIG.RANDOM_SEED}, Target: {CONFIG.PRIMARY_TARGET_VARIABLE}')


## 📥 1 & 2. Ingestión y Entendimiento de Datos
Cargamos la serie temporal del proceso de blanqueo a intervalos de 2 minutos (20,126 observaciones y 100 sensores industriales).


In [ ]:
loader = IndustrialDataLoader(config=CONFIG)
raw_df, tag_meta, unit_meta = loader.load_raw_data(use_cache=True)
print(f'Dimensiones del dataset de planta: {raw_df.shape}')
raw_df.head(5)


### 🏷️ Clasificación de Variables por Dominio Operacional
Mapeo de los 100 sensores a las 5 categorías requeridas por el enunciado:


In [ ]:
feature_class_df = DomainFeatureClassifier.classify_features(list(raw_df.columns), metadata_tags=tag_meta)
print(feature_class_df['Categoria'].value_counts())
feature_class_df.head(20)


## 🧹 3. Preparación de Datos y Prevención de Fuga (*Data Leakage*)
- Exclusión de sensores sin variación o inhabilitados.
- Aislamiento estricto de componentes aritméticos del consumo de $ClO_2$.
- Partición 80% Train / 20% Test con semilla `SEED = 2022`.
- Estandarización `StandardScaler` ajustada únicamente con `X_train`.


In [ ]:
preprocessor = IndustrialDataPreprocessor(config=CONFIG)
prep_data = preprocessor.clean_and_prepare(raw_df, target_variable=CONFIG.PRIMARY_TARGET_VARIABLE)
print(f'Conjunto de Entrenamiento: {prep_data.X_train_scaled.shape}')
print(f'Conjunto de Prueba:        {prep_data.X_test_scaled.shape}')


## 🔬 4. Modelado y Diagnósticos Estadísticos
Ajuste y validación de los 5 enfoques de regresión lineal:


In [ ]:
evaluator = ModelEvaluator(config=CONFIG)

# 4.1 OLS (MCO) con Diagnósticos
ols_model = OLSRegressionModel()
ols_model.fit(prep_data.X_train_scaled, prep_data.y_train)
evaluator.add_evaluation(ols_model.evaluate(prep_data.X_train_scaled, prep_data.y_train, prep_data.X_test_scaled, prep_data.y_test))

# 4.2 Stepwise AIC
stepwise_model = StepwiseAICRegressionModel(max_features=25, direction='both', verbose=False)
stepwise_model.fit(prep_data.X_train_scaled, prep_data.y_train)
evaluator.add_evaluation(stepwise_model.evaluate(prep_data.X_train_scaled, prep_data.y_train, prep_data.X_test_scaled, prep_data.y_test))

# 4.3 Ridge (L2 con CV)
ridge_model = RidgeRegressionModel(cv=CONFIG.CV_FOLDS)
ridge_model.fit(prep_data.X_train_scaled, prep_data.y_train)
evaluator.add_evaluation(ridge_model.evaluate(prep_data.X_train_scaled, prep_data.y_train, prep_data.X_test_scaled, prep_data.y_test))

# 4.4 Lasso (L1 con CV)
lasso_model = LassoRegressionModel(cv=CONFIG.CV_FOLDS, random_state=CONFIG.RANDOM_SEED)
lasso_model.fit(prep_data.X_train_scaled, prep_data.y_train)
evaluator.add_evaluation(lasso_model.evaluate(prep_data.X_train_scaled, prep_data.y_train, prep_data.X_test_scaled, prep_data.y_test))

# 4.5 Elastic Net (L1+L2 con CV)
enet_model = ElasticNetRegressionModel(cv=CONFIG.CV_FOLDS, random_state=CONFIG.RANDOM_SEED)
enet_model.fit(prep_data.X_train_scaled, prep_data.y_train)
evaluator.add_evaluation(enet_model.evaluate(prep_data.X_train_scaled, prep_data.y_train, prep_data.X_test_scaled, prep_data.y_test))

print('[+] Todos los modelos han sido entrenados y evaluados.')


## 📊 5. Tabla Comparativa de Modelos y Selección Final


In [ ]:
comp_df = evaluator.generate_comparison_dataframe()
evaluator.print_comparison_table()
best_model_metrics = evaluator.select_best_model()
